# Phase 2: Forecasting Diagnostics
## The Hook
While machine learning models often beat simple benchmarks, our forecasting showdown revealed a surprising result: **The Naive Baseline ($y_t = y_{t-1}$) outperformed LightGBM and Prophet on the 12-week holdout.**

This notebook performs a forensic diagnosis of *why* this occurred.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

import sys
sys.path.append('..')
from src.data.load_data import load_data
from src.data.preprocess import preprocess_data
from src.data.aggregator import aggregate_to_weekly_chain

# Load and Aggregate to Chain-Level
transactions, products = load_data('../data/raw/wcer.csv', '../data/raw/upccer.csv')
df = preprocess_data(transactions, products)
df_weekly = aggregate_to_weekly_chain(df)

df_weekly.tail()


## 1. The Weekly Chain-Level Trend
Let's zoom out and look at the entire 400-week history of Dominick's chain-level demand. Notice the spiky nature of the promotions.


In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(df_weekly['Week'], df_weekly['Sales'], color='#2ca02c', linewidth=1.5)
plt.title('Total Chain-Level Weekly Sales')
plt.xlabel('Week Number')
plt.ylabel('Total Units Sold')
plt.show()


## 2. The Saturation Reveal (The "Aha" Moment)
Our LightGBM model relied heavily on the `Promo` feature (as seen in our Feature Importance analysis). But what happens if the predictive feature loses its variance? 

Let's look at the Promo Intensity specifically focusing on the final 12 weeks (our Holdout Period).


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# Plot Promo Intensity
ax.plot(df_weekly['Week'], df_weekly['Promo'], color='#1f77b4', linewidth=2)

# Highlight the final 12 weeks (Holdout Period)
holdout_start = df_weekly['Week'].iloc[-12]
holdout_end = df_weekly['Week'].iloc[-1]

ax.axvspan(holdout_start, holdout_end, color='red', alpha=0.2, label='12-Week Holdout Period')

# Annotate the Saturation
saturation_level = df_weekly['Promo'].iloc[-12:].mean()
ax.annotate(
    f'Promo Saturation ({saturation_level:.1%} Intensity)\nZero variance in signal',
    xy=(holdout_start + 6, saturation_level),
    xytext=(holdout_start - 100, saturation_level - 0.2),
    fontsize=12,
    bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.9),
    arrowprops=dict(arrowstyle='->', color='#d62728', lw=2)
)

ax.set_title('Chain-Level Promotional Intensity Over Time', fontweight='bold')
ax.set_xlabel('Week Number')
ax.set_ylabel('% of Stores on Promotion')
ax.legend()
plt.show()


### The Implication
**Feature Degeneracy**: During the 12-week holdout, promotional intensity flatlined at over 95%. When exogenous features lose variance, the time series effectively becomes a pure random walk or highly autoregressive process. 

This visual perfectly explains our pipeline results: **In a high-saturation regime, the Naive persistence model establishes an incredibly high bar that ML struggles to beat without granular cross-sectional data (like Store-level interactions).**
